# Extract N-Back data from the csv file generated by jsPsych

Use this notebook for the N-Back task. It reads raw jsPsych CSV files and makes a cleaner table with image number, Novel/Repeated response, and accuracy.

## What can I change?

At the bottom of this notebook, change the input folder or file name so it points to your own data.


In [1]:
import os
import re
import glob
import pandas as pd
import numpy as np
import json

In [2]:
def extract_image_idx(stimulus):
    """
    Extract image number from stimulus path.
    """
    if pd.isna(stimulus):
        return None

    fname = os.path.basename(str(stimulus))
    match = re.search(r"(\d+)", fname)

    if match:
        return int(match.group(1))
    return None


def extract_n_back_file(csv_file):
    subject_id = os.path.splitext(os.path.basename(csv_file))[0].split("_")[-1]
    df = pd.read_csv(csv_file)

    # Keep only actual memorability response trials
    df = df[df["trial_type"] == "image-button-response"].copy()

    # Extract image number from stimulus filename
    df["image_idx"] = df["stimulus"].apply(extract_image_idx)

    # Clean response/correct fields
    df["response"] = pd.to_numeric(df["response"], errors="coerce")
    df["correct_response"] = pd.to_numeric(df["correct_response"], errors="coerce")
    df["correct"] = df["correct"].astype(bool)

    df["occurrence"] = df.groupby("image_idx").cumcount() + 1
    df["is_repeat"] = df["occurrence"] == 2

    df["first_trial_index"] = df.groupby("image_idx")["trial_index"].transform("first")
    df["repeat_lag"] = (df["trial_index"] - df["first_trial_index"])//2

    # This is only filled for the repeated/second occurrence
    df["correct_at_second_occurrence"] = df["correct"].where(df["is_repeat"])

    # Optional metadata from filename
    df["source_file"] = os.path.basename(csv_file)
    df["subject_id"] = subject_id

    # Keep useful columns
    out = df[
        [
            "source_file",
            "subject_id",
            "trial_index",
            "time_elapsed",
            "rt",
            "stimulus",
            "image_idx",
            "occurrence",
            "is_repeat",
            "repeat_lag",
            "response",
            "correct_response",
            "correct",
            "correct_at_second_occurrence",
        ]
    ].copy()

    return out

In [3]:
def extract_n_back_folder(input_folder, output_csv="n_back_extracted.csv"):
    all_files = glob.glob(os.path.join(input_folder, "*.csv"))

    all_data = []

    ages = []
    sexes = []

    excluded = 0
    for f in all_files:
        if "extracted" in f:
          excluded += 1
          continue

        raw_df = pd.read_csv(f)
        survey_rows = raw_df[raw_df["trial_type"] == "survey-multi-choice"]

        if len(survey_rows) > 0:

            response = survey_rows.iloc[0]["response"]


            # jsPsych stores survey responses as JSON-like strings
            if isinstance(response, str):

                age_match = re.search(r'"Age":"([^"]+)"', response)
                sex_match = re.search(r'"Gender":"([^"]+)"', response)
                print(age_match.group(1), sex_match.group(1))

                if age_match:
                    ages.append(age_match.group(1))

                if sex_match:
                    sexes.append(sex_match.group(1))

        parsed = extract_n_back_file(f)
        all_data.append(parsed)

    results = pd.concat(all_data, ignore_index=True)
    results.to_csv(output_csv, index=False)

    # ---------------------------------
    # Print survey statistics
    # ---------------------------------
    print("\n==============================")
    print("Survey Statistics")

    print(f"N participants: {len(all_files)-excluded}")

    if len(ages) > 0:
        age_counts = pd.Series(ages).value_counts()
        print("\nAge distribution:")
        for age, count in age_counts.items():
            print(f"  {age}: {count}")

    if len(sexes) > 0:
        sex_counts = pd.Series(sexes).value_counts()

        print("\nGender distribution:")
        for sex, count in sex_counts.items():
            print(f"  {sex}: {count}")

    # ---------------------------------
    # Memorability performance
    # ---------------------------------
    repeats = results[results["is_repeat"]]

    mean_acc = repeats["correct_at_second_occurrence"].mean()

    print("\n==============================")
    print("Overall Performance:")

    print(f"Mean accuracy on repeated images: {mean_acc:.3f}")

    print("\nSaved parsed data to:")
    print(output_csv)

    return results

In [ ]:
# ===============================
# STUDENTS: EDIT THIS PART BELOW
# ===============================

In [4]:
# Example: parse one file
results = extract_n_back_file("./data/n_back/n_back_298127.csv")
results.to_csv("./data/n_back/n_back_298127_extracted.csv", index=False)

In [5]:
# Example: parse all files in one folder
results = extract_n_back_folder("./data/n_back", "./data/all_data_n_back.csv")

26-30 Female

Survey Statistics
N participants: 1

Age distribution:
  26-30: 1

Gender distribution:
  Female: 1

Overall Performance:
Mean accuracy on repeated images: 1.000

Saved parsed data to:
./data/all_data_n_back.csv
